# D03 · SPI modes, daisy chains and quad SPI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/02_drivers_protocols/D03_spi_modes_daisy_chains_and_quad_spi.ipynb)

**Track:** Drivers and protocols · **Skill axis (sheet):** Drivers and protocols · **Time:** about 55 min

**Skill Hub concepts:** SPI modes; Chip-select timing and daisy chains; Quad and octal SPI flash

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("D03")

## Map: four modes, two edges

| Mode | CPOL (idle level) | CPHA | Sample on | Shift on | Example device |
|---|---|---|---|---|---|
| 0 | 0 | 0 | rising (leading) | falling | most flash, many sensors |
| 1 | 0 | 1 | falling (trailing) | rising | AS5047P encoder |
| 2 | 1 | 0 | falling (leading) | rising | |
| 3 | 1 | 1 | rising (trailing) | falling | TMC5160 stepper driver |

Rule: CPHA = 0 samples on the **leading** edge, CPHA = 1 on the **trailing** edge; CPOL decides whether leading is
rising or falling. Modes 0 and 3 both sample on rising edges, which is why a mode-0 master half-works with a mode-3
part (Storm defect: firmware uses mode 0 for both chips).

```
 Daisy chain (TMC5160, 40-bit datagrams):     MCU MOSI -> [dev0 SDI..SDO] -> [dev1 SDI..SDO] -> MCU MISO
 one CS for all; shift 40 x n bits; bits sent FIRST end up in the FARTHEST device; CS rising edge latches all.
```

| Interface | Data lines | Bits per clock | Typical use |
|---|---|---|---|
| SPI | 1 out, 1 in | 1 | sensors, drivers |
| Dual SPI | 2 | 2 | flash reads |
| Quad SPI | 4 | 4 | execute-in-place flash, displays |
| Octal SPI (often DDR) | 8 | 8 (16 with DDR) | PSRAM, fast flash |

## Task 1: decode a logic-analyzer capture
`sclk` and `data` are lists of 0/1 samples taken at the same instants. Sample `data[i]` at every index `i` where
`sclk` makes the sampling edge for this mode (`sclk[i-1] != sclk[i]`). Group bits MSB first into bytes.

In [ ]:
def spi_decode(sclk, data, cpol, cpha):
    raise NotImplementedError  # TODO

def mode_table():
    """Return {mode: (idle_level, sample_edge)} with sample_edge "rising" or "falling"."""
    raise NotImplementedError  # TODO

lab.check("decode", spi_decode, mode_table)

In [ ]:
%%writefile d03_decode.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/d03.h
// Grade it with lab.check("decode", "d03_decode.c"). For C++, name the file .cpp in both places.
#include "d03.h"

size_t spi_decode(const uint8_t *sclk, const uint8_t *data, size_t n, int cpol, int cpha, uint8_t *out, size_t out_cap) {
    (void)sclk; (void)data; (void)n; (void)cpol; (void)cpha; (void)out; (void)out_cap;
    return 0; /* TODO: on each sampling edge shift one data bit in, MSB first; emit every 8 bits */
}

void spi_mode(int mode, int *idle_level, bool *sample_rising) {
    (void)mode;
    *idle_level = 0; *sample_rising = true; /* TODO */
}

In [ ]:
# lab.check("decode", "d03_decode.c")  # uncomment after writing the C/C++ cell above

## Task 2: daisy-chain framing
`datagrams[0]` is for the device nearest the MCU. Return the integer to shift out MSB first (40 bits per device),
and split a received integer back into per-device replies (index 0 = nearest device).

In [ ]:
def chain_tx(datagrams):
    raise NotImplementedError  # TODO

def chain_rx(value, n):
    raise NotImplementedError  # TODO

lab.check("chain", chain_tx, chain_rx)

In [ ]:
%%writefile d03_chain.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/d03.h
// Grade it with lab.check("chain", "d03_chain.c"). For C++, name the file .cpp in both places.
#include "d03.h"

void chain_tx(const uint64_t *datagrams, size_t n, uint8_t *out) {
    (void)datagrams; memset(out, 0, 5 * n); /* TODO */
}

void chain_rx(const uint8_t *in, size_t n, uint64_t *datagrams) {
    (void)in; memset(datagrams, 0, n * sizeof *datagrams); /* TODO */
}

In [ ]:
# lab.check("chain", "d03_chain.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** Two TMC5160s in one chain. Clock cycles per CS-low transaction?

**M2 (Apply).** Quad SPI flash at 80 MHz, single data rate. Raw read throughput in MB/s?

**M3 (Analyze).** AS5047P at 10 MHz SPI, 16-bit frames, CS must stay high at least 350 ns between frames. Maximum
angle reads per second?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Apply).** One SPI peripheral drives a TMC5160 (mode 3) and an AS5047P (mode 1) with separate CS lines.
What must the driver do?
- a) pick mode 2 as a compromise
- b) reconfigure CPOL/CPHA for the target device while every CS is high, before each transaction
- c) nothing: the devices adapt

**K2 (Analyze).** Explain why a mode-0 master often appears to work with a mode-3 slave, where exactly it can fail,
and how you would prove it on a logic analyzer.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `d03.h`).
3. In `roboprep/checks/d03.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()